# Qwen2.5-7B full fine-tune — Yandex DataSphere (one-pass)

Self-contained DataSphere build: full fine-tuning (not LoRA) of `Qwen/Qwen2.5-7B-Instruct`
for the RAG-reliability judge (Method 1 `direct` / Method 2 `marker`).

**Requires the `g2.1` configuration (1× A100 80 GB).** Edit the config block in the
first cell, then **Run All** — every DataSphere-specific fix (driver-matched
torch, numpy pin, storage paths, transformers-5 API) is already baked in.

Before running: create a File Storage (Ресурсы проекта → Файловое хранилище →
**Активировать**), restart the compute VM so it mounts, and set `BASE` below to
its path (e.g. `/home/jupyter/filestore/<name>`).

In [ ]:
# ======================= EDIT THIS CONFIG, THEN RUN ALL =======================
BASE          = "/home/jupyter/filestore/neurodrive"   # your mounted File Storage
MODE          = "direct"          # "direct" (Method 1) or "marker" (Method 2)
USE_REAL_DATA = False             # False = dummy smoke set; True = REAL_DATA_PATH
REAL_DATA_PATH = f"{BASE}/organizers.jsonl"

PUSH_TO_HUB   = False             # True -> upload the model to the Hugging Face Hub
HUB_MODEL_ID  = ""                # e.g. "your-login/qwen2.5-7b-rag-judge-direct"
HF_TOKEN      = ""                # write token from https://huggingface.co/settings/tokens

EPOCHS, LR, MAX_SEQ_LEN         = 3, 1e-5, 2048
PER_DEVICE_BATCH, GRAD_ACCUM    = 1, 8
SEED                            = 42
SAVE_STRATEGY, SAVE_TOTAL_LIMIT = "no", 1   # "epoch" = resumable checkpoints (needs disk room)
# ==============================================================================

import os, sys, subprocess
# Set BEFORE any torch/transformers import: model cache on the big disk, no OOM fragmentation.
os.environ["HF_HOME"] = f"{BASE}/hf"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
OUTPUT_DIR = f"{BASE}/ft_{MODE}"

# Install the driver-matched stack. Version-check via metadata (does NOT import the
# modules, so it never poisons sys.modules with a stale numpy/torch before install).
from importlib.metadata import version, PackageNotFoundError
def _v(p):
    try: return version(p)
    except PackageNotFoundError: return None
def _pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

need = (_v("trl") != "1.8.0") or ((_v("numpy") or "").startswith("1.26") is False) or (_v("torch") is None)
if need:
    # torch under DataSphere's CUDA 12.2 driver (PyPI cu13 fails "driver too old";
    # base 2.0.1/cu118 is too old for trl 1.8). numpy 1.26 pinned LAST: new enough
    # for trl typing, old enough (<2) to keep base soxr/scipy/sklearn ABI-compatible.
    _pip("torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
         "--index-url", "https://download.pytorch.org/whl/cu121")
    _pip("transformers>=4.56.2", "trl==1.8.0", "accelerate>=1.4.0", "datasets==4.7.0",
         "peft>=0.8.0", "bitsandbytes>=0.44.1", "pydantic>=2.5", "sentencepiece", "psutil")
    _pip("numpy==1.26.4")
    print("Installed stack. If you had already imported torch in THIS kernel, do "
          "Kernel > Restart Kernel and Run All again; on a fresh kernel just continue.")
else:
    print("stack already present")
print("BASE:", BASE, "| MODE:", MODE, "| OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# --- Clone the repo (persisted on storage) + import format builders/parser/metrics ---
REPO_URL    = "https://github.com/aldem2k00/rag-reliability.git"
REPO_BRANCH = "qwen7b-notebook"
REPO_DIR    = f"{BASE}/rag-reliability"
import os, sys, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "-q", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR])
src = os.path.join(REPO_DIR, "src")
if src not in sys.path:
    sys.path.insert(0, src)

from rag_reliability.nb_format import build_sft_messages
from rag_reliability.dataset import load_jsonl, split_samples
from rag_reliability.parsing import parse_prediction
from rag_reliability.schema import RagSample
from rag_reliability import metrics as M
from rag_reliability.formatting import build_chat_training_record
from rag_reliability.prompts import build_direct_prompt, build_marker_prompt

DATA_PATH = REAL_DATA_PATH if USE_REAL_DATA else f"{REPO_DIR}/data/dummy.jsonl"

# Real data: if REAL_DATA_PATH is missing, build it from the organizer archive
# committed in the repo (scripts/prepare_data.py extracts the CSV from data.zip).
# If you uploaded your own organizers.jsonl to REAL_DATA_PATH, this is skipped.
if USE_REAL_DATA and not os.path.exists(REAL_DATA_PATH):
    env = dict(os.environ, PYTHONPATH=src)
    subprocess.check_call(
        [sys.executable, os.path.join(REPO_DIR, "scripts", "prepare_data.py"),
         "--input", os.path.join(REPO_DIR, "from_organizators", "data", "data.zip"),
         "--output", REAL_DATA_PATH],
        env=env)
    print("generated real dataset ->", REAL_DATA_PATH)

print("format source: repo | DATA_PATH:", DATA_PATH)


In [ ]:
# --- Hardware check: this build targets a single A100 80GB (config g2.1) ---
import torch, psutil
n = torch.cuda.device_count()
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if n else 0.0
ram = psutil.virtual_memory().total / 1e9
print(f"GPUs {n} | VRAM {vram:.0f}GB | RAM {ram:.0f}GB | "
      f"{torch.cuda.get_device_name(0) if n else 'CPU'}")
assert n >= 1 and vram >= 70, (
    "This DataSphere notebook does full fine-tuning on ONE A100 80GB (config g2.1). "
    "Select that configuration, or use the general notebook for smaller GPUs.")


In [ ]:
# --- Load data, split (stratified by reliable, seed 42), build chat records ---
from datasets import Dataset
samples = load_jsonl(DATA_PATH)
train, val, test_samples = split_samples(samples, seed=SEED)
print(f"loaded {len(samples)} -> train {len(train)} | val {len(val)} | test {len(test_samples)}")

train_ds = Dataset.from_list([build_sft_messages(s, MODE) for s in train])
val_ds   = Dataset.from_list([build_sft_messages(s, MODE) for s in val])
assert train_ds[0] == build_chat_training_record(train[0], MODE), "format drift!"
print("format symmetry OK:", train_ds[0]["messages"][1]["content"])


In [ ]:
# --- Full fine-tune: bf16 + gradient checkpointing + 8-bit AdamW on the A100 ---
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16, device_map={"": 0})
model.config.use_cache = False

cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    optim="adamw_bnb_8bit",        # 8-bit AdamW: full 7B FT fits 80GB (fp32 AdamW would OOM)
    max_length=MAX_SEQ_LEN,        # trl>=1.x renamed max_seq_length -> max_length
    bf16=True,
    gradient_checkpointing=True,
    assistant_only_loss=True,      # loss only on assistant tokens (mirrors mlx --mask-prompt)
    eval_strategy="epoch",
    per_device_eval_batch_size=1,
    logging_steps=5,
    save_strategy=SAVE_STRATEGY,
    save_total_limit=SAVE_TOTAL_LIMIT,
    report_to="none",
    seed=SEED,
)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds,
                     eval_dataset=val_ds, processing_class=tok)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)
print("training done ->", OUTPUT_DIR)


In [ ]:
# --- Evaluate: free the training model, reload, greedy-generate, parse, score ---
import gc, torch
try:
    del model, trainer
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer
build_prompt = build_direct_prompt if MODE == "direct" else build_marker_prompt
tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR, dtype=torch.bfloat16, device_map="auto")
model.eval()

@torch.no_grad()
def generate(prompt: str) -> str:
    # transformers>=5 returns a BatchEncoding here; return_dict=True + **enc keeps
    # input_ids AND attention_mask and avoids passing a dict as a positional tensor.
    enc = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                  add_generation_prompt=True, return_tensors="pt",
                                  return_dict=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=64, do_sample=False,
                         pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

preds = [parse_prediction(generate(build_prompt(s)), s.id, expect_marker=(MODE == "marker"))
         for s in test_samples]
metrics_out = M.evaluate_predictions(test_samples, preds)
print(metrics_out.model_dump_json(indent=2) if hasattr(metrics_out, "model_dump_json") else metrics_out)


In [ ]:
# --- Export: push to the Hugging Face Hub (download / reuse anywhere) ---
if PUSH_TO_HUB and HUB_MODEL_ID:
    from huggingface_hub import HfApi, create_repo
    create_repo(HUB_MODEL_ID, token=HF_TOKEN, private=True, exist_ok=True)
    HfApi().upload_folder(folder_path=OUTPUT_DIR, repo_id=HUB_MODEL_ID, token=HF_TOKEN,
                          commit_message=f"full-FT Qwen2.5-7B RAG judge ({MODE})")
    print("pushed to Hub:", f"https://huggingface.co/{HUB_MODEL_ID}")
    print(f'Load anywhere: AutoModelForCausalLM.from_pretrained("{HUB_MODEL_ID}")')
else:
    print(f"Model saved at {OUTPUT_DIR}")
    print("To export: set PUSH_TO_HUB=True + HUB_MODEL_ID + HF_TOKEN and re-run this cell,")
    print("or download the folder from the JupyterLab file manager.")
